In [26]:
from pathlib import Path

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
    RestOf, 
    SharesInflowStocks
)
from imagematerials.preprocessing import get_preprocessing_data
import numpy as np

from imagematerials.rest_of import rest_of_preprocessing
import matplotlib.pyplot as plt

from imagematerials.electricity.preprocessing import (
    get_preprocessing_data_gen,
    get_preprocessing_data_grid,
    get_preprocessing_data_stor
)

In [27]:
YEAR_START = 1971  # start year of the simulation period
YEAR_END = 2100    # end year of the calculations
YEAR_OUT = 2100    # year of output generation = last year of reporting

In [28]:
scenario_list = {
    "SSP2_M_CP":("SSP2_M_CP", None),
    "SSP2_VLLO":("SSP2_VLLO", None),
    "SSP2_VLLO_LifeTech":("SSP2_VLLO_LifeTech", ["resource_efficient"])
                 }

In [29]:
path_image_scenarios = Path("..", "data", "raw", "image")
scenario_base_path = Path("..", "data", "raw", "circular_economy_scenarios")
path_raw_data = Path("..", "data", "raw")
path_image_materials = Path("..")

In [30]:
model_runs = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running scenario: {scen_id}")
    print(f"Circular economy scenario: {circular_scen}")
    print('Current path: ', Path.cwd())

    # needed for electricity
    scen = climate_scen.split("_")[0]
    variant = "_".join(climate_scen.split("_")[1:])
    
    climate_policy_scenario_dir = path_image_scenarios / climate_scen
    if circular_scen is not None:
        circular_economy_dir = {
            scenario: scenario_base_path / scenario for scenario in circular_scen
        }
    else:
        circular_economy_dir = None

    # Define the complete timeline, including historic tail
    time_start = 1971
    complete_timeline = prism.Timeline(time_start, 2100, 1)
    simulation_timeline = prism.Timeline(1971, 2100, 1)

    # BUILDINGS
    bld_sector = get_preprocessing_data("buildings", path_raw_data, 
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs = circular_economy_dir) 
    # VEHICLES
    vhc_sector = get_preprocessing_data("vehicles", path_raw_data, 
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs = circular_economy_dir)
    prep_data_vhc = vhc_sector.prep_data
    vhc_sector = Sector('vehicles', prep_data_vhc)

    # GENERATION
    prep_data_gen = get_preprocessing_data_gen(path_image_materials, scen, variant, YEAR_START, YEAR_END, YEAR_OUT)
    sec_electr_gen = Sector("generation", prep_data_gen)
    
    # GRID
    prep_data_lines, prep_data_add = get_preprocessing_data_grid(path_image_materials, scen, variant, YEAR_START, YEAR_END, YEAR_OUT)
    sec_electr_grid_lines = Sector("grid", prep_data_lines)
    sec_electr_grid_add = Sector("grid_additional", prep_data_add)
   
    # STORAGE
    prep_data_phs, prep_data_oth_storage = get_preprocessing_data_stor(path_image_materials, scen, variant, YEAR_START, YEAR_END, YEAR_OUT)
    sec_electr_stor_phs = Sector("storage_pumped_hydropower", prep_data_phs)
    sec_electr_stor_oth = Sector("storage_other", prep_data_oth_storage)

    # REST OF SECTOR
    rest_sector = rest_of_preprocessing(path_raw_data, 
                        image_scenario_directory = climate_policy_scenario_dir, 
                        scenario = climate_scen)
    rest_sector = Sector(name='rest_of', data = rest_sector)
    

    factory = ModelFactory(
    [bld_sector, vhc_sector, sec_electr_gen, rest_sector, 
     sec_electr_grid_lines, sec_electr_grid_add, sec_electr_stor_phs, sec_electr_stor_oth], complete_timeline
    ).add(GenericStocks, ["buildings", "vehicles", "generation", "grid", "grid_additional", 'storage_pumped_hydropower'] 
    ).add(GenericMaterials,  "vehicles"
    ).add(SharesInflowStocks, "storage_other"
    ).add(MaterialIntensities, ["buildings", "generation", "grid", "grid_additional", 
                                "storage_pumped_hydropower", "storage_other"] 
    ).add(RestOf, "rest_of", input_sources={
        "gompertz_coefs": "rest_of",
        "gdp_per_capita": "rest_of",
        "population": "rest_of",
        "historic_diff_consumption_mean": "rest_of",
        "historic_diff_consumption_total": "rest_of"
}
    )
    model = factory.finish()
    model_runs[climate_scen] = model

    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)
        model.save_pkl(f'model_results/{scen_id}_model.pkl')



Running scenario: SSP2_M_CP
Circular economy scenario: None
Current path:  c:\Coding\image-materials\examples
Path to image output: ..\data\raw\image\SSP2_M_CP\EnergyServices


c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


grid_stock_lines to xarray Dataset
materials_grid_kgperkm to xarray Dataset
lifetime_grid_distr not in conversion_table
grid_stock_add to xarray Dataset
materials_grid_add_kgperunit to xarray Dataset
lifetime_grid_distr not in conversion_table
Path to image output: ..\data\raw\image\SSP2_M_CP\EnergyServices
phs_stock to xarray Dataset
phs_materials to xarray Dataset
phs_lifetime_distr not in conversion_table
oth_storage_stock to xarray Dataset
oth_storage_lifetime_distr not in conversion_table
oth_storage_shares to xarray Dataset
Running scenario: SSP2_VLLO
Circular economy scenario: None
Current path:  c:\Coding\image-materials\examples
Path to image output: ..\data\raw\image\SSP2_VLLO\EnergyServices


c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


grid_stock_lines to xarray Dataset
materials_grid_kgperkm to xarray Dataset
lifetime_grid_distr not in conversion_table
grid_stock_add to xarray Dataset
materials_grid_add_kgperunit to xarray Dataset
lifetime_grid_distr not in conversion_table
Path to image output: ..\data\raw\image\SSP2_VLLO\EnergyServices
phs_stock to xarray Dataset
phs_materials to xarray Dataset
phs_lifetime_distr not in conversion_table
oth_storage_stock to xarray Dataset
oth_storage_lifetime_distr not in conversion_table
oth_storage_shares to xarray Dataset
Running scenario: SSP2_VLLO_LifeTech
Circular economy scenario: ['resource_efficient']
Current path:  c:\Coding\image-materials\examples
Applied using resource_efficient building materials intensities for residential buildings.
implemented 'narrow' for Vehicles (lightweighting)
Path to image output: ..\data\raw\image\SSP2_VLLO_LifeTech\EnergyServices


c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


grid_stock_lines to xarray Dataset
materials_grid_kgperkm to xarray Dataset
lifetime_grid_distr not in conversion_table
grid_stock_add to xarray Dataset
materials_grid_add_kgperunit to xarray Dataset
lifetime_grid_distr not in conversion_table
Path to image output: ..\data\raw\image\SSP2_VLLO_LifeTech\EnergyServices
phs_stock to xarray Dataset
phs_materials to xarray Dataset
phs_lifetime_distr not in conversion_table
oth_storage_stock to xarray Dataset
oth_storage_lifetime_distr not in conversion_table
oth_storage_shares to xarray Dataset
Using Gompertz coefficients for resource efficiency measures
Mismatch in coordinates with dimension 'material' with data array 'historic_diff_consumption_mean' having different coordinates than previously assumed in '['gompertz_coefs']'.New: [np.str_('aluminium'), np.str_('cement'), np.str_('clay'), np.str_('copper'), np.str_('limestone'), np.str_('sand'), np.str_('steel')]

Old:[np.str_('Aluminium'), np.str_('Cement'), np.str_('Clay'), np.str_('Coppe

AlignmentError: cannot align objects with join='exact' where index/labels/sizes are not equal along these coordinates (dimensions): 'material' ('material',)

In [ ]:
# save totals of materials for rest-of fitting

from imagematerials.rest_of.preprocessing.sum_sectors_image_materials import sum_inflows_for_output

materials_dict_metal = {
    'Steel' : 'Steel',
    'Aluminium' : 'Aluminium',
    'Copper' : 'Cu',
}

materials_dict_nmm = {
    'Cement' : 'Cement',
    'Sand' : 'Sand'
}

materials_dict_sand = {
    'Sand' : 'Sand'
}
total_material_metals = sum_inflows_for_output(model, materials_dict_metal, 'metals')
total_material_nmm = sum_inflows_for_output(model, materials_dict_nmm, 'nmm')
total_material_sand = sum_inflows_for_output(model, materials_dict_sand, 'nmm')

In [34]:
model_runs['SSP2_M_CP'].generation.keys()

dict_keys(['lifetimes', 'stocks', 'material_intensities', 'knowledge_graph', 'set_unit_flexible', 'outflow_by_cohort', 'inflow', 'stock_by_cohort', 'stock_by_cohort_materials', 'inflow_materials', 'outflow_by_cohort_materials'])